In [3]:
import pandas as pd
import networkx as nx
import time

In [4]:
# 1. Iniciamos el cronómetro
inicio_carga = time.time() # Dado a que las aristas son 6 millones, el tiempo de carga puede ser significativo, por lo que hemos querido medirlo

# 2. Cargar los datos. 
print("Cargando datos...")
df = pd.read_csv('large_twitch_edges.csv') 

# 3. Convertir el DataFrame a un grafo de NetworkX
print("Creando grafo...")
G = nx.from_pandas_edgelist(df, source='numeric_id_1', target='numeric_id_2')

# 4. Paramos el cronómetro
fin_carga = time.time()

# 5. Mostramos los resultados
tiempo_total = fin_carga - inicio_carga
print("-----------------------------------")
print(f"Tiempo total: {tiempo_total:.2f} segundos")
print(f"Nodos cargados: {G.number_of_nodes()}")
print(f"Aristas cargadas: {G.number_of_edges()}")

Cargando datos...
Creando grafo...
-----------------------------------
Tiempo total: 10.43 segundos
Nodos cargados: 168114
Aristas cargadas: 6797557


In [11]:
# Subgrafo con los primeros 20k nodos (ids 0..19999)
nodos_20k = [n for n in range(0, 20000) if G.has_node(n)]
G_20k = G.subgraph(nodos_20k).copy()

print(f"Nodos en G_20k: {G_20k.number_of_nodes()}")
print(f"Aristas en G_20k: {G_20k.number_of_edges()}")

edges_20k = nx.to_pandas_edgelist(G_20k)
edges_20k.head()

Nodos en G_20k: 20000
Aristas en G_20k: 92798


,source,target
0,0,10441
1,0,13048
2,0,10464
3,1,8079
4,1,6250


In [23]:
def centralidades_csv(G_sub, csv_path="datos_relacionales_20k.csv"):
    dc = nx.degree_centrality(G_sub)
    bc = nx.betweenness_centrality(G_sub, k=5000) # Para acelerar el cálculo, se puede usar una muestra de nodos para el cálculo de la centralidad de intermediación
    cc = nx.closeness_centrality(G_sub)
    clustering = nx.clustering(G_sub)
    triangles = nx.triangles(G_sub)
    pageRank = nx.pagerank(G_sub)

    df_out = pd.DataFrame({
        "nodo": list(G_sub.nodes()),
        "degree_centrality": [dc[n] for n in G_sub.nodes()],
        "betweenness_centrality": [bc[n] for n in G_sub.nodes()],
        "closeness_centrality": [cc[n] for n in G_sub.nodes()],
        "clustering": [clustering[n] for n in G_sub.nodes()],
        "triangles": [triangles[n] for n in G_sub.nodes()],
        "pageRank": [pageRank[n] for n in G_sub.nodes()]
    })
    df_out.to_csv(csv_path, index=False)
    return df_out

centralidades_df = centralidades_csv(G_20k)
centralidades_df.head()

,nodo,degree_centrality,betweenness_centrality,closeness_centrality,clustering,triangles,pageRank
0,0,0.00015,4.298175e-07,0.191578,0.000000,0,0.000023
1,1,0.00210,4.900130e-04,0.298126,0.111498,96,0.000174
2,2,0.00060,1.203790e-05,0.235950,0.015152,1,0.000056
3,3,0.00010,3.569050e-07,0.211446,0.000000,0,0.000016
4,4,0.00030,8.741067e-05,0.227000,0.133333,2,0.000051


In [25]:
def unificar_features_y_relaciones(
    features_path="large_twitch_features.csv",
    relaciones_path="datos_relacionales_20k.csv",
    salida_path="features_relaciones_20k.csv",
    limite=20000,
):
    features_df = pd.read_csv(features_path).head(limite).reset_index(drop=True)
    relaciones_df = pd.read_csv(relaciones_path).head(limite).reset_index(drop=True)

    if "nodo" in relaciones_df.columns:
        relaciones_df = relaciones_df.drop(columns=["nodo"])

    unificado_df = pd.concat([features_df, relaciones_df], axis=1)
    #Eliminar filas donde todas las centralidades sean 0, ya que no aportan información para la clasificación
    centralidades_cols = [
        "degree_centrality",
        "betweenness_centrality",
        "closeness_centrality",
    ]
    unificado_df = unificado_df[
        (unificado_df[centralidades_cols] != 0).any(axis=1)
    ]

    unificado_df.to_csv(salida_path, index=False)
    return unificado_df

unificado_df = unificar_features_y_relaciones()
unificado_df.head()

,views,mature,life_time,created_at,updated_at,numeric_id,dead_account,language,affiliate,degree_centrality,betweenness_centrality,closeness_centrality,clustering,triangles,pageRank
0,7879,1,969,2016-02-16,2018-10-12,0,0,EN,1,0.00015,4.298175e-07,0.191578,0.000000,0,0.000023
1,500,0,2699,2011-05-19,2018-10-08,1,0,EN,0,0.00210,4.900130e-04,0.298126,0.111498,96,0.000174
2,382502,1,3149,2010-02-27,2018-10-12,2,0,EN,1,0.00060,1.203790e-05,0.235950,0.015152,1,0.000056
3,386,0,1344,2015-01-26,2018-10-01,3,0,EN,0,0.00010,3.569050e-07,0.211446,0.000000,0,0.000016
4,2486,0,1784,2013-11-22,2018-10-11,4,0,EN,0,0.00030,8.741067e-05,0.227000,0.133333,2,0.000051


Fallo en GPU, usando CPU. Detalle: module 'cugraph' has no attribute 'closeness_centrality'


KeyboardInterrupt: 